# Can we predict whether a student passes the closing quiz?
Target: `passed_quiz`. Follows `checklist.md`.

Design notes (decided before looking at any model score):
- There is no date/row-order column, so the split cannot be temporal. Students appear in many rows (about 7 each), so **whole students** are held out (checklist 1) and CV uses `GroupKFold` on student (2).
- Dropped: `student_key` (identifier). `school_code` is a nominal code and is one-hot encoded, not treated as a number.
- Assumption to flag: `session_minutes` and `hints_used` are recorded during the session, so they are only available if we predict at the end of tutoring, just before the quiz. If the model must predict at session start, only the pre-session candidate (M1) is valid. Both are named as candidates below.

In [ ]:
import numpy as np, pandas as pd
from sklearn.base import clone
from sklearn.model_selection import GroupShuffleSplit, GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score, balanced_accuracy_score, accuracy_score, log_loss
df = pd.read_csv('data.csv')
print(df.shape, df.isna().sum().sum(), 'missing;', df.student_key.nunique(), 'students; pass rate', df.passed_quiz.mean().round(3))

## 1. Test set: 20% of students, set aside before any modelling

In [ ]:
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=0)
tr_idx, te_idx = next(gss.split(df, groups=df.student_key))
train, test = df.iloc[tr_idx].reset_index(drop=True), df.iloc[te_idx].reset_index(drop=True)
assert not set(train.student_key) & set(test.student_key)
print(len(train), len(test), train.passed_quiz.mean().round(3), test.passed_quiz.mean().round(3))

## 2. Candidates (named before any scoring)
- **M0** baseline: predicts the training pass rate.
- **M1** logistic regression, pre-session features: `prior_score`, `enrol_year`, `school_code`.
- **M2** logistic regression, M1 + `session_minutes`, `hints_used`.
- **M3** gradient boosting (`HistGradientBoostingClassifier`, depth 3, lr 0.05, 200 iters), M2 features.
- **M4** gradient boosting, depth 3, lr 0.05, 100 iters (stronger regularisation variant of M3).

Selection metric: cross-validated ROC AUC (threshold-free). The decision threshold is then chosen on out-of-fold predictions to maximise balanced accuracy. Preprocessing (scaling, one-hot) lives inside the pipelines, so it is fit within each fold (3).

In [ ]:
pre = ['prior_score','enrol_year']; inses = ['session_minutes','hints_used']
def prep(num, cat=('school_code',)):
    return ColumnTransformer([('n', StandardScaler(), list(num)), ('c', OneHotEncoder(handle_unknown='ignore'), list(cat))])
def hgb(n):
    # school_code is the last column (index 4) of the feature list and is treated as categorical
    return HistGradientBoostingClassifier(max_depth=3, learning_rate=0.05, max_iter=n, categorical_features=[4], random_state=0)
full = pre+inses+['school_code']
cands = {
 'M0 baseline': (Pipeline([('m', DummyClassifier(strategy='prior'))]), pre),
 'M1 LR pre-session': (Pipeline([('p', prep(pre)), ('m', LogisticRegression(max_iter=1000))]), pre+['school_code']),
 'M2 LR +session': (Pipeline([('p', prep(pre+inses)), ('m', LogisticRegression(max_iter=1000))]), full),
 'M3 HGB 200': (Pipeline([('m', hgb(200))]), full),
 'M4 HGB 100': (Pipeline([('m', hgb(100))]), full),
}

## 3. Grouped cross-validation on the training rows only

In [ ]:
cv = GroupKFold(n_splits=5)
oof, rows = {}, []
y = train.passed_quiz.values
for name,(pipe,cols) in cands.items():
    p = np.zeros(len(train)); aucs=[]
    for a,b in cv.split(train, y, groups=train.student_key):
        m = clone(pipe).fit(train.iloc[a][cols], y[a])
        p[b] = m.predict_proba(train.iloc[b][cols])[:,1]
        aucs.append(roc_auc_score(y[b], p[b]))
    oof[name]=p
    rows.append((name, np.mean(aucs), np.std(aucs), log_loss(y,p)))
cvres = pd.DataFrame(rows, columns=['model','cv_auc_mean','cv_auc_sd','oof_logloss']).set_index('model')
cvres

## 4. Choice, written down before the test set is touched

In [ ]:
best = cvres.cv_auc_mean.idxmax()
p = oof[best]
ths = np.linspace(0.2,0.7,101)
bal = [balanced_accuracy_score(y, p>=t) for t in ths]
thr = float(ths[int(np.argmax(bal))])
print('chosen model:', best, '| threshold:', round(thr,3), '| CV balanced acc at threshold:', round(max(bal),3))
print('pre-session M1 CV AUC:', round(cvres.loc['M1 LR pre-session','cv_auc_mean'],3))

## 5. Refit on all training rows, score the test set once
This is the only test-set evaluation; the number below is the headline.

In [ ]:
pipe, cols = cands[best]
final = clone(pipe).fit(train[cols], y)
pt = final.predict_proba(test[cols])[:,1]; yt = test.passed_quiz.values
# bootstrap over test students for an interval
rng = np.random.default_rng(0); sids = test.student_key.values; u = np.unique(sids)
groups = {s:np.where(sids==s)[0] for s in u}
bs=[]
for _ in range(500):
    ix = np.concatenate([groups[s] for s in rng.choice(u, len(u))])
    bs.append(roc_auc_score(yt[ix], pt[ix]))
print(f'TEST AUC {roc_auc_score(yt,pt):.3f} (95% student-bootstrap CI {np.percentile(bs,2.5):.3f}-{np.percentile(bs,97.5):.3f})')
print(f'TEST balanced acc {balanced_accuracy_score(yt, pt>=thr):.3f}, accuracy {accuracy_score(yt, pt>=thr):.3f}, majority-class accuracy {max(yt.mean(),1-yt.mean()):.3f}')

## Conclusion
See the printed test numbers above. Read them with these caveats:
- Splits are by student because no time column exists; if the model will be applied to future sessions, performance on later data may differ (pass rate drifts slightly by `enrol_year`).
- The best model uses `session_minutes` and `hints_used`, which are only known once the session is under way. For prediction at session start, the pre-session model M1 (CV AUC in the table) is the relevant one.
- `prior_score` varies slightly within a student (std about 0.3 vs 11.7 between students) even though it is a placement score; this looked like measurement noise and was not related to the outcome.